# ETL: Silver → Gold Layer (Data Warehouse)

Este notebook realiza a transformação dos dados normalizados da camada Silver para a camada Gold, criando um Data Warehouse com modelo dimensional (Star Schema).

## Processo:
1. **Extract**: Extrai dados da tabela Silver no PostgreSQL
2. **Transform**: Cria dimensões e tabela fato (Star Schema)
3. **Load**: Carrega dimensões e fato no schema `gold`

---

## 1. Importações e Configuração

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine, text
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configurações do Banco de Dados
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'sinistros_prf')
DB_USER = os.getenv('DB_USER', 'prf_user')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'prf_pass')

# String de Conexão
CONNECTION_STRING = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print("Configurações carregadas:")
print(f"Host: {DB_HOST}:{DB_PORT}")
print(f"Database: {DB_NAME}")

Configurações carregadas:
Host: localhost:5432
Database: sinistros_prf


In [ ]:
def build_dimensions_and_fact(df: pd.DataFrame):
    """
    Cria dimensões e fato a partir do DataFrame da Silver.
    
    Retorna:
    - dict com 6 dimensões (dim_temporal, dim_localizacao, dim_categorizacao, dim_via, dim_pessoa, dim_veiculo)
    - DataFrame fat_sinistro
    """
    print("Criando modelo dimensional (Star Schema)...\n")
    
    # DIMENSÃO TEMPORAL
    print("Criando dim_temporal...")
    dim_temporal = (
        df[["data", "ano", "hora", "dia_semana", "periodo", "periodo_semana"]]
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(srk_tmp=lambda x: x.index + 1)
        .rename(columns={
            "data": "tmp_dta",
            "ano": "tmp_ano",
            "hora": "tmp_hra",
            "dia_semana": "tmp_dsm",
            "periodo": "tmp_per",
            "periodo_semana": "tmp_psm"
        })
    )
    print(f"{len(dim_temporal):,} registros únicos")
    
    # DIMENSÃO LOCALIZAÇÃO
    print("Criando dim_localizacao...")
    dim_localizacao = (
        df[
            [
                "uf",
                "localidade",
                "regiao",
                "municipio",
                "rodovia",
                "rodovia_numero",
                "quilometro",
                "latitude",
                "longitude",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(srk_loc=lambda x: x.index + 1)
        .rename(columns={
            "uf": "loc_uf",
            "localidade": "loc_ldd",
            "regiao": "loc_reg",
            "municipio": "loc_mun",
            "rodovia": "loc_rod",
            "rodovia_numero": "loc_nrd",
            "quilometro": "loc_km",
            "latitude": "loc_lat",
            "longitude": "loc_lng"
        })
    )
    print(f"{len(dim_localizacao):,} registros únicos")
    
    # DIMENSÃO CATEGORIZAÇÃO
    print("Criando dim_categorizacao...")
    dim_categorizacao = (
        df[
            [
                "sinistro_tipo",
                "sinistro_causa",
                "sinistro_causa_principal",
                "sinistro_ordem_tipo",
                "gravidade",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(srk_cat=lambda x: x.index + 1)
        .rename(columns={
            "sinistro_tipo": "cat_tip",
            "sinistro_causa": "cat_cau",
            "sinistro_causa_principal": "cat_cap",
            "sinistro_ordem_tipo": "cat_ord",
            "gravidade": "cat_grv"
        })
    )
    print(f"{len(dim_categorizacao):,} registros únicos")
    
    # DIMENSÃO VIA
    print("Criando dim_via...")
    dim_via = (
        df[
            [
                "condicao_meteorologica",
                "via_tipo",
                "via_tracado",
                "via_sentido",
                "uso_solo",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(srk_via=lambda x: x.index + 1)
        .rename(columns={
            "condicao_meteorologica": "via_cmt",
            "via_tipo": "via_tip",
            "via_tracado": "via_tra",
            "via_sentido": "via_sen",
            "uso_solo": "via_uso"
        })
    )
    print(f"{len(dim_via):,} registros únicos")
    
    # DIMENSÃO PESSOA
    print("Criando dim_pessoa...")
    dim_pessoa = (
        df[
            [
                "envolvido_tipo",
                "envolvido_sexo",
                "envolvido_idade",
                "faixa_etaria_ano",
                "faixa_etaria_classe",
                "estado_fisico",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(srk_pes=lambda x: x.index + 1)
        .rename(columns={
            "envolvido_tipo": "pes_tip",
            "envolvido_sexo": "pes_sex",
            "envolvido_idade": "pes_idd",
            "faixa_etaria_ano": "pes_fxa",
            "faixa_etaria_classe": "pes_fxc",
            "estado_fisico": "pes_esf"
        })
    )
    print(f"{len(dim_pessoa):,} registros únicos")
    
    # DIMENSÃO VEÍCULO
    print("Criando dim_veiculo...")
    dim_veiculo = (
        df[
            [
                "veiculo_tipo",
                "veiculo_marca_modelo",
                "veiculo_ano_fabricacao",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(srk_vei=lambda x: x.index + 1)
        .rename(columns={
            "veiculo_tipo": "vei_tip",
            "veiculo_marca_modelo": "vei_mrc",
            "veiculo_ano_fabricacao": "vei_ano"
        })
    )
    print(f"{len(dim_veiculo):,} registros únicos\n")
    
    # CONSTRUÇÃO DA FATO
    print("Construindo tabela fato...")
    
    # Preparar DataFrames originais para merge
    df_tmp = df[["data", "ano", "hora", "dia_semana", "periodo", "periodo_semana"]].copy()
    df_tmp.columns = ["tmp_dta", "tmp_ano", "tmp_hra", "tmp_dsm", "tmp_per", "tmp_psm"]
    
    df_loc = df[["uf", "localidade", "regiao", "municipio", "rodovia", "rodovia_numero", "quilometro", "latitude", "longitude"]].copy()
    df_loc.columns = ["loc_uf", "loc_ldd", "loc_reg", "loc_mun", "loc_rod", "loc_nrd", "loc_km", "loc_lat", "loc_lng"]
    
    df_cat = df[["sinistro_tipo", "sinistro_causa", "sinistro_causa_principal", "sinistro_ordem_tipo", "gravidade"]].copy()
    df_cat.columns = ["cat_tip", "cat_cau", "cat_cap", "cat_ord", "cat_grv"]
    
    df_via = df[["condicao_meteorologica", "via_tipo", "via_tracado", "via_sentido", "uso_solo"]].copy()
    df_via.columns = ["via_cmt", "via_tip", "via_tra", "via_sen", "via_uso"]
    
    df_pes = df[["envolvido_tipo", "envolvido_sexo", "envolvido_idade", "faixa_etaria_ano", "faixa_etaria_classe", "estado_fisico"]].copy()
    df_pes.columns = ["pes_tip", "pes_sex", "pes_idd", "pes_fxa", "pes_fxc", "pes_esf"]
    
    df_vei = df[["veiculo_tipo", "veiculo_marca_modelo", "veiculo_ano_fabricacao"]].copy()
    df_vei.columns = ["vei_tip", "vei_mrc", "vei_ano"]
    
    # Criar fato com joins
    fato = pd.concat([
        df[["sinistro_id", "ilesos", "feridos_leves", "feridos_graves", "feridos", "mortos"]],
        df_tmp,
        df_loc,
        df_cat,
        df_via,
        df_pes,
        df_vei
    ], axis=1)
    
    # Merge com dimensões para obter SRKs
    fato = (
        fato.merge(dim_temporal, on=["tmp_dta", "tmp_ano", "tmp_hra", "tmp_dsm", "tmp_per", "tmp_psm"], how="left")
        .merge(dim_localizacao, on=["loc_uf", "loc_ldd", "loc_reg", "loc_mun", "loc_rod", "loc_nrd", "loc_km", "loc_lat", "loc_lng"], how="left")
        .merge(dim_categorizacao, on=["cat_tip", "cat_cau", "cat_cap", "cat_ord", "cat_grv"], how="left")
        .merge(dim_via, on=["via_cmt", "via_tip", "via_tra", "via_sen", "via_uso"], how="left")
        .merge(dim_pessoa, on=["pes_tip", "pes_sex", "pes_idd", "pes_fxa", "pes_fxc", "pes_esf"], how="left")
        .merge(dim_veiculo, on=["vei_tip", "vei_mrc", "vei_ano"], how="left")
    )
    
    # Selecionar e renomear colunas finais da fato
    fat_sinistro = fato[[
        "sinistro_id",
        "srk_tmp",
        "srk_loc",
        "srk_cat",
        "srk_via",
        "srk_pes",
        "srk_vei",
        "ilesos",
        "feridos_leves",
        "feridos_graves",
        "feridos",
        "mortos"
    ]].rename(columns={
        "sinistro_id": "srk_sns",
        "ilesos": "fat_ils",
        "feridos_leves": "fat_fle",
        "feridos_graves": "fat_fgr",
        "feridos": "fat_fer",
        "mortos": "fat_mrt"
    })
    
    print(f"{len(fat_sinistro):,} registros na fato\n")
    
    print("Modelo dimensional criado com sucesso!")
    print(f"\nResumo:")
    print(f"6 Dimensões criadas")
    print(f"1 Tabela Fato criada")
    print(f"Total: {len(dim_temporal) + len(dim_localizacao) + len(dim_categorizacao) + len(dim_via) + len(dim_pessoa) + len(dim_veiculo) + len(fat_sinistro):,} registros")
    
    dimensions = {
        "dim_temporal": dim_temporal,
        "dim_localizacao": dim_localizacao,
        "dim_categorizacao": dim_categorizacao,
        "dim_via": dim_via,
        "dim_pessoa": dim_pessoa,
        "dim_veiculo": dim_veiculo,
    }
    
    return dimensions, fat_sinistro

print("Função de transformação criada!")

Função de transformação criada!


## 2. Extract - Carregamento dos Dados da Camada Silver

In [7]:
# Criar engine do SQLAlchemy
engine = create_engine(
    CONNECTION_STRING,
    pool_pre_ping=True,
    pool_size=5,
    max_overflow=10
)

print("Testando conexão ao PostgreSQL...")
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version()"))
        version = result.fetchone()[0]
        print(f"Conexão estabelecida!")
        print(f"PostgreSQL: {version.split(',')[0]}\n")
        
        # Extrair dados da tabela Silver
        print("Extraindo dados da camada Silver...")
        query = "SELECT * FROM dl.tb_sinistros_silver"
        df_silver = pd.read_sql(query, engine)
        
        print(f"{len(df_silver):,} registros extraídos")
        print(f"Colunas: {len(df_silver.columns)}")
except Exception as e:
    print(f"Erro na conexão: {e}")

Testando conexão ao PostgreSQL...
Conexão estabelecida!
PostgreSQL: PostgreSQL 15.15 on x86_64-pc-linux-musl

Extraindo dados da camada Silver...
981,790 registros extraídos
Colunas: 45
981,790 registros extraídos
Colunas: 45


## 3. Transform - Criação do Modelo Dimensional (Star Schema)

In [8]:
# Criar dimensões e fato
dimensions, fato_sinistros = build_dimensions_and_fact(df_silver.copy())

Criando modelo dimensional (Star Schema)...

Criando dim_temporal...
14,413 registros únicos
Criando dim_localizacao...
14,413 registros únicos
Criando dim_localizacao...
94,351 registros únicos
Criando dim_categorizacao...
94,351 registros únicos
Criando dim_categorizacao...
16,238 registros únicos
Criando dim_via...
16,238 registros únicos
Criando dim_via...
5,549 registros únicos
Criando dim_pessoa...
5,549 registros únicos
Criando dim_pessoa...
2,117 registros únicos
Criando dim_veiculo...
2,117 registros únicos
Criando dim_veiculo...
230,692 registros únicos

Construindo tabela fato...
230,692 registros únicos

Construindo tabela fato...
981,790 registros na fato

Modelo dimensional criado com sucesso!

Resumo:
6 Dimensões criadas
1 Tabela Fato criada
Total: 1,345,150 registros
981,790 registros na fato

Modelo dimensional criado com sucesso!

Resumo:
6 Dimensões criadas
1 Tabela Fato criada
Total: 1,345,150 registros


## 4. Load - Carregamento no PostgreSQL (Schema dw)

Carregar todas as dimensões e tabela fato no banco de dados no schema `dw`.

In [9]:
# Criar schema dw se não existir e limpar tabelas antigas
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS dw"))
    
    # Dropar tabelas antigas com CASCADE para remover dependências
    tables_to_drop = [
        'fat_sinistro',
        'dim_temporal',
        'dim_localizacao', 
        'dim_categorizacao',
        'dim_via',
        'dim_pessoa',
        'dim_veiculo'
    ]
    
    for table in tables_to_drop:
        try:
            conn.execute(text(f"DROP TABLE IF EXISTS dw.{table} CASCADE"))
        except:
            pass
    
print("Schema 'dw' criado/verificado")

# Carregar dimensões
print("Carregando dimensões no banco...\n")

for dim_name, dim_df in dimensions.items():
    print(f"Carregando {dim_name}...")
    dim_df.to_sql(
        name=dim_name,
        con=engine,
        schema='dw',
        if_exists='replace',
        index=False,
        method='multi',
        chunksize=5000
    )
    print(f"{len(dim_df):,} registros")

# Carregar fato
print(f"\nCarregando fat_sinistro...")
fato_sinistros.to_sql(
    name='fat_sinistro',
    con=engine,
    schema='dw',
    if_exists='replace',
    index=False,
    method='multi',
    chunksize=5000
)
print(f"{len(fato_sinistros):,} registros")

print("\nTODAS AS TABELAS CARREGADAS COM SUCESSO!")

Schema 'dw' criado/verificado
Carregando dimensões no banco...

Carregando dim_temporal...
14,413 registros
Carregando dim_localizacao...
14,413 registros
Carregando dim_localizacao...
94,351 registros
Carregando dim_categorizacao...
94,351 registros
Carregando dim_categorizacao...
16,238 registros
Carregando dim_via...
16,238 registros
Carregando dim_via...
5,549 registros
Carregando dim_pessoa...
5,549 registros
Carregando dim_pessoa...
2,117 registros
Carregando dim_veiculo...
2,117 registros
Carregando dim_veiculo...
230,692 registros

Carregando fat_sinistro...
230,692 registros

Carregando fat_sinistro...
981,790 registros

TODAS AS TABELAS CARREGADAS COM SUCESSO!
981,790 registros

TODAS AS TABELAS CARREGADAS COM SUCESSO!


## 5. Verificação - Consultas no Data Warehouse

In [10]:
# Verificar totais de cada tabela
print("Verificando dados carregados:\n")

tables = ['dim_temporal', 'dim_localizacao', 'dim_categorizacao', 'dim_via', 'dim_pessoa', 'dim_veiculo', 'fat_sinistro']

for table in tables:
    query = f"SELECT COUNT(*) as total FROM dw.{table}"
    result = pd.read_sql(query, engine)
    print(f"{table}: {result['total'][0]:,} registros")

Verificando dados carregados:

dim_temporal: 14,413 registros
dim_localizacao: 94,351 registros
dim_categorizacao: 16,238 registros
dim_via: 5,549 registros
dim_pessoa: 2,117 registros
dim_veiculo: 230,692 registros
fat_sinistro: 981,790 registros


In [11]:
# Exemplo de consulta analítica: Top 10 UFs com mais sinistros
query = """
SELECT 
    dl.loc_uf,
    dl.loc_ldd,
    dl.loc_reg,
    COUNT(*) as total_sinistros,
    SUM(f.fat_mrt) as total_mortos,
    SUM(f.fat_fer) as total_feridos
FROM dw.fat_sinistro f
JOIN dw.dim_localizacao dl ON f.srk_loc = dl.srk_loc
GROUP BY dl.loc_uf, dl.loc_ldd, dl.loc_reg
ORDER BY total_sinistros DESC
LIMIT 10
"""

print("Top 10 UFs com mais sinistros:\n")
pd.read_sql(query, engine)

Top 10 UFs com mais sinistros:



,loc_uf,loc_ldd,loc_reg,total_sinistros,total_mortos,total_feridos
0,MG,Minas Gerais,Sudeste,132928,7060.0,58250.0
1,PR,Paraná,Sul,118236,5291.0,47161.0
2,SC,Santa Catarina,Sul,88498,3209.0,39699.0
3,RS,Rio Grande do Sul,Sul,63533,2491.0,26693.0
4,BA,Bahia,Nordeste,56990,4030.0,23973.0
5,SP,São Paulo,Sudeste,54258,1252.0,23097.0
6,RJ,Rio de Janeiro,Sudeste,53467,1310.0,25409.0
7,GO,Goiás,Centro-Oeste,51315,2561.0,19457.0
8,MT,Mato Grosso,Centro-Oeste,41352,1925.0,13804.0
9,PE,Pernambuco,Nordeste,41261,2318.0,19038.0


## 6. Resumo Final

ETL Silver → Gold concluído com sucesso!

**Data Warehouse criado com:**
- 6 tabelas dimensionais (dim_temporal, dim_localizacao, dim_categoriazacao, dim_via, dim_pessoa, dim_veiculo)
- 1 tabela fato (fat_sinistro)
- Modelo Star Schema otimizado para análises

**Próximos passos:**
1. Execute queries analíticas usando `data_layer/gold/consultas.sql
2. Consulte `data_layer/gold/mnemonico.md` para referência de nomenclatura
3. Consulte o Painel dos dados em: 